# Frozen CLIP Vision experiment — separate from the working prototype
Use a **new Colab GPU runtime**. Run cells sequentially. Existing baseline checkpoints, feature caches, processed data and Streamlit files are read-only.
The experiment adds CLIP image-only and cached DistilBERT + CLIP fusion. It never downloads Fakeddit, re-splits data, fine-tunes encoders or modifies the app.
CLIP's projected image features are **512-dimensional**; cached DistilBERT is **768-dimensional**. All classification heads encode their actual input dimensions.
References: [CLIP API](https://huggingface.co/docs/transformers/model_doc/clip), [model card](https://huggingface.co/openai/clip-vit-base-patch32).


## 1. Dependencies and GPU check
Keep Colab's matched torch/torchvision CUDA packages. Only Python dependencies and pretrained CLIP weights may be downloaded.

In [ ]:
%pip -q install "transformers==4.57.1" "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" "Pillow>=10,<13" safetensors
import torch, torchvision
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU is required."
print(torch.cuda.get_device_name(0), torch.__version__, torchvision.__version__)


## 2. Mount Drive and install embedded source into a separate temporary directory

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, json, shutil, gc, hashlib
drive.mount('/content/drive')
WORK = Path('/content/fakeddit_clip_code')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
SOURCE_FILES = {'dataset.py': '"""Read existing paired data without rewriting CSVs or downloading images."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport pandas as pd\nfrom PIL import Image\n\nSIZES = {"train": 3000, "validate": 500, "test": 500}\nLABELS = {0: "Fake", 1: "Real"}\n\n\ndef resolve_image(root, raw):\n    root = Path(root).resolve()\n    parts = str(raw).replace("\\\\", "/").split("/")\n    # Relocate even absolute Windows paths by the images/ suffix.\n    if "images" not in parts:\n        raise ValueError(f"Image path has no images directory: {raw}")\n    relative = Path(*parts[parts.index("images"):])\n    target = (root / relative).resolve()\n    if not target.is_relative_to(root):\n        raise ValueError(f"Unsafe image path: {raw}")\n    return target\n\n\ndef extract_zip(archive, destination):\n    destination = Path(destination).resolve()\n    destination.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as z:\n        for entry in z.infolist():\n            name = entry.filename.replace("\\\\", "/")\n            target = (destination / name).resolve()\n            if not target.is_relative_to(destination) or ":" in name:\n                raise ValueError(f"Unsafe ZIP member: {name}")\n            if (entry.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError("ZIP symlinks are not supported")\n        for entry in z.infolist():\n            target = destination / entry.filename.replace("\\\\", "/")\n            if entry.is_dir():\n                target.mkdir(parents=True, exist_ok=True)\n            else:\n                target.parent.mkdir(parents=True, exist_ok=True)\n                with z.open(entry) as src, target.open("wb") as dst:\n                    import shutil\n                    shutil.copyfileobj(src, dst)\n    matches = list(destination.rglob("train_processed.csv"))\n    if len(matches) != 1:\n        raise ValueError("ZIP must contain exactly one processed dataset")\n    return matches[0].parent\n\n\ndef validate(root, report_path):\n    frames, report, errors = {}, {"label_mapping": LABELS, "splits": {}}, []\n    for split, expected in SIZES.items():\n        df = pd.read_csv(Path(root) / f"{split}_processed.csv", dtype={"id": str}, keep_default_na=False)\n        required = ["id", "clean_title", "2_way_label", "local_image_path"]\n        if not set(required).issubset(df.columns):\n            raise ValueError(f"Missing columns in {split}: {set(required) - set(df.columns)}")\n        missing = {c: int(df[c].astype(str).str.strip().eq("").sum()) for c in required}\n        duplicates = df.loc[df.id.duplicated(), "id"].tolist()\n        if len(df) != expected or any(missing.values()) or duplicates:\n            errors.append(f"{split}: count/missing/duplicate failure")\n        numeric = pd.to_numeric(df["2_way_label"], errors="coerce")\n        if not numeric.isin([0, 1]).all():\n            errors.append(f"{split}: labels must be 0 or 1")\n        if "6_way_label" in df:\n            six = pd.to_numeric(df["6_way_label"], errors="coerce")\n            if not six.isin(range(6)).all() or not ((six == 0) == (numeric == 1)).all():\n                errors.append(f"{split}: binary labels inconsistent with 6-way true category")\n        paths, hashes, bad = [], [], []\n        for row in df.itertuples(index=False):\n            try:\n                p = resolve_image(root, row.local_image_path)\n                with Image.open(p) as im:\n                    im.verify()\n                with Image.open(p) as im:\n                    im.convert("RGB").load()\n                hashes.append(hashlib.sha256(p.read_bytes()).hexdigest())\n                paths.append(str(p))\n            except Exception as exc:\n                bad.append({"id": row.id, "error": str(exc)})\n                paths.append("")\n                hashes.append("")\n        if bad:\n            errors.append(f"{split}: {len(bad)} missing/corrupt images")\n        df["resolved_path"], df["image_sha256"] = paths, hashes\n        frames[split] = df\n        report["splits"][split] = {"count": len(df), "classes": df["2_way_label"].value_counts().to_dict(),\n                                  "missing": missing, "duplicate_ids": duplicates, "bad_images": bad}\n    names = list(frames)\n    report["overlaps"] = {}\n    for i, a in enumerate(names):\n        for b in names[i + 1:]:\n            overlap = sorted(set(frames[a].id) & set(frames[b].id))\n            report["overlaps"][f"{a}/{b}"] = overlap\n            if overlap:\n                errors.append(f"{a}/{b}: overlapping sample IDs")\n    report["errors"] = errors\n    Path(report_path).parent.mkdir(parents=True, exist_ok=True)\n    Path(report_path).write_text(json.dumps(report, indent=2), encoding="utf-8")\n    if errors:\n        raise ValueError(f"Validation failed; see {report_path}: {errors}")\n    return frames, report\n\n\nif __name__ == "__main__":\n    _, report = validate("data", "results/dataset_validation.json")\n    print(json.dumps(report, indent=2))\n', 'models.py': '"""Shared Colab/Windows model contract. Backbones are always frozen."""\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom PIL import ImageOps\nfrom torchvision.models import resnet50, ResNet50_Weights\nfrom transformers import AutoModel, AutoTokenizer\n\nENCODER_CONFIG = {"text_model": "distilbert-base-uncased", "max_length": 128,\n                  "pooling": "attention_mask_mean", "image_weights": "IMAGENET1K_V2",\n                  "text_dim": 768, "image_dim": 2048, "version": 1}\nHEAD_CONFIG = {"text": {"dimensions": [768, 256, 2], "dropout": 0.3},\n               "image": {"dimensions": [2048, 256, 2], "dropout": 0.3},\n               "fusion": {"text_projection": [768, 256], "image_projection": [2048, 256],\n                          "dimensions": [512, 128, 2], "dropout": 0.3}, "version": 1}\n\n\nclass Head(nn.Module):\n    def __init__(self, kind):\n        super().__init__()\n        if kind not in ("text", "image", "fusion"):\n            raise ValueError(kind)\n        self.kind = kind\n        if kind == "fusion":\n            self.text = nn.Linear(768, 256)\n            self.image = nn.Linear(2048, 256)\n            self.classifier = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(.3), nn.Linear(128, 2))\n        else:\n            self.classifier = nn.Sequential(nn.Linear(768 if kind == "text" else 2048, 256),\n                                            nn.ReLU(), nn.Dropout(.3), nn.Linear(256, 2))\n\n    def forward(self, text, image):\n        x = torch.cat((self.text(text), self.image(image)), dim=1) if self.kind == "fusion" else (text if self.kind == "text" else image)\n        return self.classifier(x)\n\n\ndef masked_mean(hidden, mask):\n    mask = mask.unsqueeze(-1).to(hidden.dtype)\n    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)\n\n\nclass Encoders:\n    def __init__(self, assets, device="cpu", download=False):\n        self.device = torch.device(device)\n        assets = Path(assets)\n        text_dir = assets / "distilbert"\n        source = ENCODER_CONFIG["text_model"] if download else str(text_dir)\n        self.tokenizer = AutoTokenizer.from_pretrained(source, local_files_only=not download)\n        self.text = AutoModel.from_pretrained(source, local_files_only=not download)\n        weights = ResNet50_Weights.IMAGENET1K_V2\n        self.image = resnet50(weights=weights if download else None)\n        if download:\n            assets.mkdir(parents=True, exist_ok=True)\n            self.tokenizer.save_pretrained(text_dir)\n            self.text.save_pretrained(text_dir)\n            torch.save(self.image.state_dict(), assets / "resnet50.pt")\n        else:\n            self.image.load_state_dict(torch.load(assets / "resnet50.pt", map_location="cpu", weights_only=True))\n        self.image.fc = nn.Identity()\n        self.transform = weights.transforms()\n        for model in (self.text, self.image):\n            model.requires_grad_(False).eval().to(self.device)\n\n    @torch.inference_mode()\n    def encode_text(self, titles):\n        tokens = self.tokenizer(titles, padding=True, truncation=True, max_length=128, return_tensors="pt").to(self.device)\n        return masked_mean(self.text(**tokens).last_hidden_state, tokens["attention_mask"]).float()\n\n    @torch.inference_mode()\n    def encode_images(self, images):\n        batch = torch.stack([self.transform(ImageOps.exif_transpose(im).convert("RGB")) for im in images])\n        return self.image(batch.to(self.device)).float()\n', 'pipeline.py': '"""GPU-only bulk extraction/training; persistent, fingerprinted recovery files."""\nimport hashlib\nimport json\nimport os\nimport pickle\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom PIL import Image\nfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix\nfrom models import ENCODER_CONFIG, Head\n\n\ndef atomic_save(obj, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    torch.save(obj, tmp)\n    os.replace(tmp, path)\n\n\ndef write_json(obj, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(".json.tmp")\n    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")\n    os.replace(tmp, path)\n\n\ndef fingerprint(frames, config):\n    records = {s: df[["id", "clean_title", "2_way_label", "image_sha256"]].to_dict("records") for s, df in frames.items()}\n    return hashlib.sha256(json.dumps({"data": records, "config": config}, sort_keys=True).encode()).hexdigest()[:20]\n\n\ndef gpu():\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select Colab Runtime > Change runtime type > GPU, then rerun. Bulk CPU work is disabled.")\n    return torch.device("cuda")\n\n\ndef extract(frames, encoders, cache, batch_size=32):\n    gpu()\n    cache = Path(cache)\n    result = {}\n    for split, df in frames.items():\n        # Separate modality chunks let a completed text batch survive an image failure.\n        features = {}\n        for modality, dim in (("text", 768), ("image", 2048)):\n            chunks = []\n            for start in range(0, len(df), batch_size):\n                batch = df.iloc[start:start + batch_size]\n                ids = batch.id.tolist()\n                labels = torch.tensor(batch["2_way_label"].tolist(), dtype=torch.long)\n                path = cache / split / modality / f"{start:06d}-{len(batch):04d}.pt"\n                part = None\n                if path.exists():\n                    try:\n                        saved = torch.load(path, map_location="cpu", weights_only=True)\n                        if (saved["ids"] == ids and torch.equal(saved["labels"], labels)\n                                and saved["features"].shape == (len(batch), dim)\n                                and torch.isfinite(saved["features"]).all()):\n                            part = saved\n                    except (OSError, RuntimeError, EOFError, ValueError, KeyError, pickle.UnpicklingError):\n                        print("Recomputing unreadable chunk:", path)\n                if part is None:\n                    if modality == "text":\n                        embedding = encoders.encode_text(batch.clean_title.tolist())\n                    else:\n                        images = []\n                        for filename in batch.resolved_path:\n                            with Image.open(filename) as im:\n                                images.append(im.convert("RGB"))\n                        embedding = encoders.encode_images(images)\n                    part = {"ids": ids, "labels": labels, "features": embedding.cpu()}\n                    if not torch.isfinite(part["features"]).all():\n                        raise ValueError("Non-finite embeddings")\n                    atomic_save(part, path)\n                chunks.append(part["features"])\n            features[modality] = torch.cat(chunks)\n            print(f"{split}/{modality}: {len(df)} cached features ready")\n        result[split] = {**features, "labels": torch.tensor(df["2_way_label"].tolist(), dtype=torch.long), "ids": df.id.tolist()}\n    return result\n\n\ndef train(kind, features, run, config, run_id):\n    device = gpu()\n    run = Path(run)\n    last_path = run / "recovery" / f"{kind}.pt"\n    best_path = run / "models" / f"{kind}.pt"\n    seed = config["seed"]\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    model = Head(kind).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])\n    loss_fn = nn.CrossEntropyLoss()\n    start, best, stale, history, best_state = 0, float("inf"), 0, [], None\n    if last_path.exists():\n        state = torch.load(last_path, map_location="cpu", weights_only=True)\n        if state["run_id"] != run_id:\n            raise ValueError("Recovery fingerprint mismatch")\n        model.load_state_dict(state["state_dict"])\n        optimizer.load_state_dict(state["optimizer"])\n        for opt in optimizer.state.values():\n            for key, value in opt.items():\n                if isinstance(value, torch.Tensor):\n                    opt[key] = value.to(device)\n        start, best, stale, history = state["epoch"] + 1, state["best"], state["stale"], state["history"]\n        best_state = state["best_state"]\n        torch.set_rng_state(state["rng"])\n        torch.cuda.set_rng_state_all(state["cuda_rng"])\n        print(f"{kind}: recovering after epoch {start}")\n    data = {s: {k: v.to(device) for k, v in features[s].items() if k != "ids"} for s in ("train", "validate")}\n    for epoch in range(start, config["epochs"]):\n        if stale >= config["patience"]:\n            break\n        model.train()\n        order = torch.randperm(len(data["train"]["labels"]), device=device)\n        total = 0.0\n        for idx in order.split(config["batch_size"]):\n            d = data["train"]\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(d["text"][idx], d["image"][idx]), d["labels"][idx])\n            loss.backward()\n            optimizer.step()\n            total += loss.item() * len(idx)\n        model.eval()\n        with torch.inference_mode():\n            d = data["validate"]\n            val = loss_fn(model(d["text"], d["image"]), d["labels"]).item()\n        if not np.isfinite(val):\n            raise ValueError("Non-finite validation loss")\n        history.append({"epoch": epoch + 1, "train_loss": total / len(order), "val_loss": val})\n        if val < best:\n            best, stale = val, 0\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n        # One recovery file contains both last and best states, avoiding split-file recovery races.\n        atomic_save({"run_id": run_id, "epoch": epoch, "state_dict": model.state_dict(),\n                     "optimizer": optimizer.state_dict(), "best_state": best_state, "best": best,\n                     "stale": stale, "history": history, "rng": torch.get_rng_state(),\n                     "cuda_rng": torch.cuda.get_rng_state_all()}, last_path)\n        print(f"{kind} epoch {epoch + 1}: train={history[-1][\'train_loss\']:.4f} val={val:.4f}")\n    if best_state is None:\n        raise ValueError("No trained checkpoint available; epochs must be positive")\n    atomic_save({"state_dict": best_state, "kind": kind, "run_id": run_id}, best_path)\n    write_json(history, run / "results" / f"{kind}_history.json")\n    return history\n\n\ndef evaluate(features, run, run_id):\n    device = gpu()\n    run = Path(run)\n    test = features["test"]\n    assert len(test["ids"]) == 500, "Use the unchanged 500-example test split"\n    y = test["labels"].numpy()\n    metrics, predictions = {}, {"id": test["ids"], "label": y.tolist()}\n    for kind in ("text", "image", "fusion"):\n        checkpoint = torch.load(run / "models" / f"{kind}.pt", map_location="cpu", weights_only=True)\n        assert checkpoint["run_id"] == run_id\n        model = Head(kind).to(device).eval()\n        model.load_state_dict(checkpoint["state_dict"])\n        with torch.inference_mode():\n            probabilities = model(test["text"].to(device), test["image"].to(device)).softmax(1).cpu().numpy()\n        pred = probabilities.argmax(1)\n        p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", pos_label=0, zero_division=0)\n        metrics[kind] = {"accuracy": float(accuracy_score(y, pred)), "precision": float(p), "recall": float(r),\n                         "f1": float(f), "roc_auc": float(roc_auc_score(y == 0, probabilities[:, 0])),\n                         "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist()}\n        predictions[f"{kind}_prediction"] = pred.tolist()\n        predictions[f"{kind}_fake_probability"] = probabilities[:, 0].tolist()\n    result = {"run_id": run_id, "test_count": 500, "positive_class": "Fake (0)", "models": metrics,\n              "fusion_outperforms_both": {metric: all(metrics["fusion"][metric] > metrics[k][metric] for k in ("text", "image"))\n                                          for metric in ("accuracy", "f1")}}\n    write_json(result, run / "results" / "metrics.json")\n    import pandas as pd\n    pd.DataFrame(predictions).to_csv(run / "results" / "predictions.csv", index=False)\n    import matplotlib\n    matplotlib.use("Agg")  # Export figures without a desktop/Tk dependency.\n    import matplotlib.pyplot as plt\n    pd.DataFrame(metrics).T[["accuracy", "precision", "recall", "f1", "roc_auc"]].plot.bar(ylim=(0, 1), figsize=(10, 5))\n    plt.tight_layout()\n    plt.savefig(run / "results" / "comparison.png")\n    plt.close()\n    for kind in metrics:\n        fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n        matrix = np.array(metrics[kind]["confusion_matrix"])\n        axes[0].imshow(matrix, cmap="Blues")\n        for (i, j), value in np.ndenumerate(matrix):\n            axes[0].text(j, i, str(value), ha="center")\n        axes[0].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"], xlabel="Predicted", ylabel="Actual")\n        history = pd.read_json(run / "results" / f"{kind}_history.json")\n        axes[1].plot(history.epoch, history.train_loss, label="Train")\n        axes[1].plot(history.epoch, history.val_loss, label="Validation")\n        axes[1].legend()\n        axes[1].set(xlabel="Epoch", ylabel="Cross-entropy")\n        fig.suptitle(kind)\n        fig.tight_layout()\n        fig.savefig(run / "results" / f"{kind}.png")\n        plt.close(fig)\n    return result\n', 'clip_models.py': '"""Experimental CLIP heads and offline encoders; independent of the production app."""\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom PIL import ImageOps\nfrom transformers import AutoModel, AutoTokenizer, CLIPImageProcessor, CLIPVisionModelWithProjection\n\nCLIP_ID = "openai/clip-vit-base-patch32"\nFEATURE_SPEC = {"model": CLIP_ID, "output": "image_embeds", "projection_dim": 512,\n                "l2_normalize": True, "processor": "CLIPImageProcessor", "version": 1}\n\n\nclass ClipHead(nn.Module):\n    def __init__(self, kind, text_dim=768, image_dim=512, dropout=0.3):\n        super().__init__()\n        if kind not in ("clip_image", "clip_fusion"):\n            raise ValueError(f"Unknown experimental head: {kind}")\n        if text_dim <= 0 or image_dim <= 0:\n            raise ValueError("Feature dimensions must be positive")\n        self.spec = dict(kind=kind, text_dim=text_dim, image_dim=image_dim, dropout=dropout)\n        self.kind = kind\n        if kind == "clip_fusion":\n            self.text_projection = nn.Linear(text_dim, 256)\n            self.image_projection = nn.Linear(image_dim, 256)\n            self.classifier = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 2))\n        else:\n            self.classifier = nn.Sequential(nn.Linear(image_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 2))\n\n    def forward(self, text, image):\n        if image.ndim != 2 or image.shape[1] != self.spec["image_dim"]:\n            raise ValueError("Wrong CLIP image feature dimension")\n        if self.kind == "clip_fusion":\n            if text.ndim != 2 or text.shape != (len(image), self.spec["text_dim"]):\n                raise ValueError("Wrong text feature dimension or pairing")\n            image = torch.cat([self.text_projection(text), self.image_projection(image)], dim=1)\n        return self.classifier(image)\n\n\nclass ClipEncoder:\n    def __init__(self, source, device="cpu", revision=None, local=True):\n        kwargs = {"local_files_only": local}\n        if revision is not None:\n            kwargs["revision"] = revision\n        self.processor = CLIPImageProcessor.from_pretrained(str(source), **kwargs)\n        self.model = CLIPVisionModelWithProjection.from_pretrained(str(source), **kwargs)\n        self.model.requires_grad_(False).eval().to(device)\n        self.device = torch.device(device)\n        self.dimension = self.model.config.projection_dim\n        if self.dimension != FEATURE_SPEC["projection_dim"]:\n            raise ValueError("Unexpected CLIP projection size")\n\n    def save(self, path):\n        self.model.save_pretrained(path, safe_serialization=True)\n        self.processor.save_pretrained(path)\n\n    @torch.inference_mode()\n    def encode(self, images):\n        images = [ImageOps.exif_transpose(im).convert("RGB") for im in images]\n        inputs = self.processor(images=images, return_tensors="pt")\n        image = self.model(pixel_values=inputs.pixel_values.to(self.device)).image_embeds.float()\n        return F.normalize(image, p=2, dim=-1)\n\n\nclass CachedTextEncoder:\n    """Use exactly the baseline\'s exported DistilBERT and masked-mean preprocessing."""\n    def __init__(self, source, device="cpu"):\n        self.tokenizer = AutoTokenizer.from_pretrained(str(source), local_files_only=True)\n        self.model = AutoModel.from_pretrained(str(source), local_files_only=True)\n        self.model.requires_grad_(False).eval().to(device)\n        self.device = torch.device(device)\n\n    @torch.inference_mode()\n    def encode(self, titles):\n        tokens = self.tokenizer(titles, padding=True, truncation=True, max_length=128, return_tensors="pt").to(self.device)\n        hidden = self.model(**tokens).last_hidden_state\n        mask = tokens["attention_mask"].unsqueeze(-1).to(hidden.dtype)\n        return ((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).float()\n', 'clip_experiment.py': '"""Isolated GPU experiment. Baseline modules, caches, and checkpoints are read-only."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport pickle\nimport shutil\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, roc_auc_score, confusion_matrix\n\nfrom models import Head, ENCODER_CONFIG\nfrom pipeline import atomic_save, write_json, fingerprint, gpu\nfrom clip_models import ClipEncoder, ClipHead, FEATURE_SPEC, CLIP_ID\n\nEXPERIMENT_VERSION = 1\nKINDS = ("clip_image", "clip_fusion")\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:20]\n\n\ndef sha_file(path):\n    with Path(path).open("rb") as file:\n        return hashlib.file_digest(file, "sha256").hexdigest()\n\n\ndef guard_output(output, *protected):\n    output = Path(output).resolve()\n    for path in protected:\n        path = Path(path).resolve()\n        if output.is_relative_to(path) or path.is_relative_to(output):\n            raise ValueError(f"Experiment output must be separate from protected path: {path}")\n    return output\n\n\ndef read_baseline_config(run, frames):\n    run = Path(run)\n    config = json.loads((run / "config.json").read_text(encoding="utf-8"))\n    if config["encoders"] != ENCODER_CONFIG or config["labels"] != {"0": "Fake", "1": "Real"}:\n        raise ValueError("Baseline encoder or label contract differs from this experiment")\n    expected = fingerprint(frames, config["encoders"])\n    if config["feature_id"] != expected:\n        raise ValueError("Baseline dataset fingerprint mismatch; do not mix different processed datasets")\n    provenance = {"config": config, "checkpoint_sha256": {}}\n    for kind in ("text", "image", "fusion"):\n        path = run / "models" / f"{kind}.pt"\n        saved = torch.load(path, map_location="cpu", weights_only=True)\n        if saved["run_id"] != config["run_id"] or saved["kind"] != kind:\n            raise ValueError("Baseline checkpoints do not belong to the selected run")\n        provenance["checkpoint_sha256"][kind] = sha_file(path)\n    return config, provenance\n\n\ndef read_cached_modality(folder, frame, dimension):\n    """Join by ID and validate labels; never trust chunk order or row positions."""\n    rows = {}\n    paths = sorted(Path(folder).glob("*.pt"))\n    if not paths:\n        raise FileNotFoundError(f"No existing features in {folder}. Restore the original Drive feature cache; this experiment does not regenerate it.")\n    for path in paths:\n        part = torch.load(path, map_location="cpu", weights_only=True)\n        ids, labels, features = part["ids"], part["labels"], part["features"]\n        if (features.shape != (len(ids), dimension) or labels.shape != (len(ids),)\n                or labels.dtype != torch.long or not torch.isfinite(features).all()):\n            raise ValueError(f"Invalid cached feature dimensions/values: {path}")\n        for index, sample_id in enumerate(ids):\n            label, embedding = int(labels[index]), features[index].float()\n            if sample_id in rows:\n                old_label, old_embedding = rows[sample_id]\n                if label != old_label or not torch.allclose(embedding, old_embedding, atol=1e-5, rtol=1e-5):\n                    raise ValueError(f"Conflicting duplicate cache ID: {sample_id}")\n            rows[sample_id] = (label, embedding)\n    if set(rows) != set(frame.id) or len(frame.id) != len(set(frame.id)):\n        raise ValueError(f"Missing/extra IDs in {folder}; cannot preserve split alignment")\n    expected = frame["2_way_label"].astype(int).tolist()\n    if [rows[sample_id][0] for sample_id in frame.id] != expected:\n        raise ValueError(f"Cached labels disagree with CSV in {folder}")\n    return torch.stack([rows[sample_id][1] for sample_id in frame.id])\n\n\ndef load_baseline_features(cache_root, feature_id, frames):\n    result = {}\n    for split, frame in frames.items():\n        folder = Path(cache_root) / feature_id / split\n        result[split] = {\n            "ids": frame.id.tolist(), "labels": torch.tensor(frame["2_way_label"].tolist(), dtype=torch.long),\n            "text": read_cached_modality(folder / "text", frame, 768),\n            "image": read_cached_modality(folder / "image", frame, 2048),\n        }\n    return result\n\n\ndef prepare_clip(output, revision="main"):\n    """Resolve once to an immutable HF revision; future recovery is offline."""\n    gpu()\n    output = Path(output)\n    request = {"model": CLIP_ID, "requested_revision": revision}\n    root = output / "encoders" / digest(request)\n    marker = root / "READY.json"\n    if marker.exists():\n        manifest = json.loads(marker.read_text())\n        if manifest["request"] != request or manifest["feature_spec"] != FEATURE_SPEC:\n            raise ValueError("CLIP asset manifest mismatch")\n        encoder = ClipEncoder(root, "cuda", local=True)\n    else:\n        from huggingface_hub import model_info\n        commit = model_info(CLIP_ID, revision=revision).sha\n        encoder = ClipEncoder(CLIP_ID, "cuda", revision=commit, local=False)\n        root.mkdir(parents=True, exist_ok=True)\n        encoder.save(root)\n        manifest = {"request": request, "resolved_revision": commit, "feature_spec": FEATURE_SPEC,\n                    "sha256": {p.name: sha_file(p) for p in root.iterdir()\n                               if p.is_file() and not p.name.endswith(".tmp")}}\n        write_json(manifest, marker)\n    for name, expected in manifest["sha256"].items():\n        if sha_file(root / name) != expected:\n            raise ValueError(f"CLIP asset changed: {name}; restore it or use a new experiment output directory")\n    return encoder, root, manifest\n\n\ndef extract_clip(frames, encoder, folder, feature_id, batch_size=32):\n    gpu()\n    if batch_size < 1:\n        raise ValueError("Feature batch size must be positive")\n    folder = Path(folder) / feature_id\n    result = {}\n    for split, frame in frames.items():\n        # Fixed shards are independent of GPU batch size: lowering the batch size retains completed work.\n        chunks = []\n        for start in range(0, len(frame), 128):\n            batch = frame.iloc[start:start + 128]\n            labels = torch.tensor(batch["2_way_label"].tolist(), dtype=torch.long)\n            ids = batch.id.tolist()\n            path = folder / split / f"{start:06d}.pt"\n            saved = None\n            if path.exists():\n                try:\n                    saved = torch.load(path, map_location="cpu", weights_only=True)\n                    if (saved["feature_id"] != feature_id or saved["ids"] != ids or not torch.equal(saved["labels"], labels)\n                            or saved["features"].shape != (len(ids), encoder.dimension)\n                            or not torch.isfinite(saved["features"]).all()):\n                        saved = None\n                except (OSError, RuntimeError, EOFError, ValueError, KeyError, pickle.UnpicklingError):\n                    saved = None\n            if saved is None:\n                embeddings = []\n                for offset in range(0, len(batch), batch_size):\n                    images = []\n                    for filename in batch.iloc[offset:offset + batch_size].resolved_path:\n                        with Image.open(filename) as image:\n                            images.append(image.copy())\n                    embeddings.append(encoder.encode(images).cpu())\n                feature = torch.cat(embeddings)\n                if feature.shape != (len(ids), encoder.dimension) or not torch.isfinite(feature).all():\n                    raise ValueError("Invalid CLIP embeddings")\n                saved = {"feature_id": feature_id, "ids": ids, "labels": labels, "features": feature}\n                atomic_save(saved, path)\n            chunks.append(saved["features"])\n        result[split] = torch.cat(chunks)\n        print(f"CLIP {split}: {len(frame)} features ready")\n    return result\n\n\ndef paired_features(baseline, visual):\n    return {split: {**data, "image": visual[split]} for split, data in baseline.items()}\n\n\ndef cache_hashes(features):\n    return {split: {modality: hashlib.sha256(data[modality].contiguous().numpy().tobytes()).hexdigest()\n                    for modality in ("text", "image")} for split, data in features.items()}\n\n\ndef binary_metrics(labels, probabilities):\n    y = np.asarray(labels)\n    probabilities = np.asarray(probabilities)\n    if probabilities.shape != (len(y), 2) or not np.isfinite(probabilities).all():\n        raise ValueError("Invalid binary probabilities")\n    pred = probabilities.argmax(1)\n    precision, recall, f1, _ = precision_recall_fscore_support(y, pred, average="binary", pos_label=0, zero_division=0)\n    return {"accuracy": float(accuracy_score(y, pred)), "precision": float(precision), "recall": float(recall),\n            "f1": float(f1), "macro_f1": float(f1_score(y, pred, labels=[0, 1], average="macro", zero_division=0)),\n            "roc_auc": float(roc_auc_score(y == 0, probabilities[:, 0])) if len(set(y)) == 2 else None,\n            "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist()}\n\n\n@torch.inference_mode()\ndef probabilities(model, data, device):\n    model.to(device).eval()\n    output = []\n    for start in range(0, len(data["labels"]), 128):\n        logits = model(data["text"][start:start + 128].to(device), data["image"][start:start + 128].to(device))\n        output.append(logits.softmax(1).cpu())\n    return torch.cat(output).numpy()\n\n\ndef train_trial(kind, data, root, config, run_id):\n    """Validation macro-F1 selects epochs; validation loss breaks exact ties."""\n    device = gpu()\n    spec = dict(kind=kind, text_dim=data["train"]["text"].shape[1], image_dim=data["train"]["image"].shape[1], dropout=config["dropout"])\n    trial_id = digest({"run_id": run_id, "spec": spec, "training": config})\n    root = Path(root) / "trials" / kind / trial_id\n    last = root / "recovery.pt"\n    root.mkdir(parents=True, exist_ok=True)\n    if config["epochs"] < 1 or config["patience"] < 1 or config["batch_size"] < 1:\n        raise ValueError("Epochs, patience, and batch size must be positive")\n    torch.manual_seed(config["seed"])\n    torch.cuda.manual_seed_all(config["seed"])\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    model = ClipHead(**spec).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])\n    start, stale, best_score, best_loss, best_state, history = 0, 0, -1., float("inf"), None, []\n    if last.exists():\n        state = torch.load(last, map_location="cpu", weights_only=True)\n        if state["trial_id"] != trial_id or state["run_id"] != run_id:\n            raise ValueError("Trial recovery mismatch")\n        model.load_state_dict(state["state_dict"])\n        optimizer.load_state_dict(state["optimizer"])\n        # AdamW maps moment buffers to the parameter device during load_state_dict.\n        start, stale = state["epoch"] + 1, state["stale"]\n        best_score, best_loss, best_state, history = state["best_score"], state["best_loss"], state["best_state"], state["history"]\n        torch.set_rng_state(state["rng"])\n        torch.cuda.set_rng_state_all(state["cuda_rng"])\n    # Test tensors are never accessed by training or trial selection.\n    train = {k: data["train"][k].to(device) for k in ("text", "image", "labels")}\n    val = {k: data["validate"][k].to(device) for k in ("text", "image", "labels")}\n    loss_fn = nn.CrossEntropyLoss()\n    for epoch in range(start, config["epochs"]):\n        if stale >= config["patience"]:\n            break\n        model.train()\n        order = torch.randperm(len(train["labels"]), device=device)\n        total = 0.\n        for ids in order.split(config["batch_size"]):\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(train["text"][ids], train["image"][ids]), train["labels"][ids])\n            loss.backward()\n            optimizer.step()\n            total += loss.item() * len(ids)\n        model.eval()\n        with torch.inference_mode():\n            logits = model(val["text"], val["image"])\n            val_loss = loss_fn(logits, val["labels"]).item()\n            metric = binary_metrics(val["labels"].cpu().numpy(), logits.softmax(1).cpu().numpy())\n        if not np.isfinite(val_loss):\n            raise ValueError("Non-finite validation loss")\n        score = metric["macro_f1"]\n        if (score, -val_loss) > (best_score, -best_loss):\n            best_score, best_loss, stale = score, val_loss, 0\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n        history.append({"epoch": epoch + 1, "train_loss": total / len(order), "validation_loss": val_loss, "validation_macro_f1": score})\n        atomic_save({"run_id": run_id, "trial_id": trial_id, "epoch": epoch, "stale": stale,\n                     "best_score": best_score, "best_loss": best_loss, "best_state": best_state, "history": history,\n                     "state_dict": model.state_dict(), "optimizer": optimizer.state_dict(),\n                     "rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all()}, last)\n        print(f"{kind} {trial_id} epoch {epoch + 1}: validation macro-F1={score:.4f}")\n    if best_state is None:\n        raise ValueError("No completed training epoch")\n    checkpoint = {"run_id": run_id, "trial_id": trial_id, "spec": spec, "state_dict": best_state,\n                  "training": config, "validation_macro_f1": best_score, "validation_loss": best_loss}\n    atomic_save(checkpoint, root / "best.pt")\n    write_json(history, root / "history.json")\n    return root / "best.pt"\n\n\ndef select_trials(data, run, configs, run_id):\n    run = Path(run)\n    selected = {}\n    for kind in KINDS:\n        candidates = [train_trial(kind, data, run, cfg, run_id) for cfg in configs]\n        records = [torch.load(path, map_location="cpu", weights_only=True) for path in candidates]\n        index = max(range(len(records)), key=lambda i: (records[i]["validation_macro_f1"], -records[i]["validation_loss"]))\n        saved = records[index]\n        atomic_save(saved, run / "models" / f"{kind}.pt")\n        selected[kind] = {k: saved[k] for k in ("trial_id", "spec", "training", "validation_macro_f1", "validation_loss")}\n        selected[kind]["all_trials"] = [{k: r[k] for k in ("trial_id", "training", "validation_macro_f1", "validation_loss")} for r in records]\n    write_json({"selection_metric": "validation macro-F1; ties: lower validation loss, then grid order", "selected": selected}, run / "selection.json")\n    return selected\n\n\ndef compare_models(baseline, clip_data, baseline_run, run, run_id, splits=("validate",)):\n    device = gpu()\n    run, baseline_run = Path(run), Path(baseline_run)\n    (run / "results").mkdir(parents=True, exist_ok=True)\n    results = {}\n    base_config = json.loads((baseline_run / "config.json").read_text())\n    for split in splits:\n        if split not in ("validate", "test") or len(baseline[split]["ids"]) != 500:\n            raise ValueError("Comparison requires the unchanged 500-example validation/test split")\n        if baseline[split]["ids"] != clip_data[split]["ids"] or not torch.equal(baseline[split]["labels"], clip_data[split]["labels"]):\n            raise ValueError("Comparison split pairing mismatch")\n        rows, metrics = {}, {}\n        for name, old_kind in (("distilbert", "text"), ("resnet50", "image"), ("distilbert_resnet50", "fusion"), ("clip_image", None), ("clip_fusion", None)):\n            if old_kind:\n                saved = torch.load(baseline_run / "models" / f"{old_kind}.pt", map_location="cpu", weights_only=True)\n                if saved["run_id"] != base_config["run_id"] or saved["kind"] != old_kind:\n                    raise ValueError("Invalid baseline checkpoint")\n                model, data = Head(old_kind), baseline[split]\n            else:\n                saved = torch.load(run / "models" / f"{name}.pt", map_location="cpu", weights_only=True)\n                if saved["run_id"] != run_id or saved["spec"]["kind"] != name:\n                    raise ValueError("Invalid CLIP checkpoint")\n                model, data = ClipHead(**saved["spec"]), clip_data[split]\n            model.load_state_dict(saved["state_dict"])\n            probs = probabilities(model, data, device)\n            metrics[name] = binary_metrics(data["labels"].numpy(), probs)\n            rows[f"{name}_prediction"] = probs.argmax(1)\n            rows[f"{name}_fake_probability"] = probs[:, 0]\n            del model\n        pd.DataFrame({"id": baseline[split]["ids"], "label": baseline[split]["labels"].tolist(), **rows}).to_csv(run / "results" / f"{split}_predictions.csv", index=False)\n        result = {"run_id": run_id, "split": split, "count": 500, "positive_class": "Fake (0)",\n                  "confusion_order": ["Fake", "Real"], "models": metrics}\n        write_json(result, run / "results" / f"{split}_metrics.json")\n        pd.DataFrame(metrics).T.drop(columns="confusion_matrix").to_csv(run / "results" / f"{split}_comparison.csv")\n        results[split] = result\n    return results\n\n\ndef promotion_gate(validation, minimum_gain=0.0):\n    if validation["split"] != "validate" or minimum_gain < 0:\n        raise ValueError("Promotion uses validation only and nonnegative minimum gain")\n    metrics = validation["models"]\n    comparisons = {"clip_image": "resnet50", "clip_fusion": "distilbert_resnet50"}\n    return {"criterion": "strict validation macro-F1 improvement", "minimum_gain": minimum_gain,\n            "candidates": {kind: {"baseline": old, "delta_macro_f1": metrics[kind]["macro_f1"] - metrics[old]["macro_f1"],\n                                   "eligible": metrics[kind]["macro_f1"] - metrics[old]["macro_f1"] > minimum_gain}\n                           for kind, old in comparisons.items()},\n            "note": "Eligibility is an observed validation improvement, not statistical significance. Test metrics do not change this decision."}\n\n\ndef plot_comparison(run, split):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    run = Path(run)\n    result = json.loads((run / "results" / f"{split}_metrics.json").read_text())\n    frame = pd.DataFrame(result["models"]).T.drop(columns="confusion_matrix").astype(float)\n    frame.plot.bar(ylim=(0, 1), figsize=(12, 5))\n    plt.tight_layout()\n    plt.savefig(run / "results" / f"{split}_comparison.png")\n    plt.close()\n    fig, axes = plt.subplots(1, len(result["models"]), figsize=(17, 3.5))\n    for ax, (name, metric) in zip(axes, result["models"].items()):\n        matrix = np.array(metric["confusion_matrix"])\n        ax.imshow(matrix, cmap="Blues")\n        for (i, j), count in np.ndenumerate(matrix):\n            ax.text(j, i, str(count), ha="center")\n        ax.set(title=name, xlabel="Predicted", ylabel="Actual", xticks=[0, 1], yticks=[0, 1], xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])\n    fig.tight_layout()\n    fig.savefig(run / "results" / f"{split}_confusion_matrices.png")\n    plt.close(fig)\n\n\ndef export_bundle(run, clip_assets, text_assets, destination, config, source_dir):\n    """Separate clip_artifacts package; no production files or baseline checkpoint copies."""\n    run, destination = Path(run), Path(destination)\n    for name in ("validate_metrics.json", "test_metrics.json", "promotion.json"):\n        if not (run / "results" / name).exists():\n            raise FileNotFoundError(f"Complete evaluation before export: {name}")\n    bundle = destination / "clip_artifacts"\n    bundle.mkdir(parents=True, exist_ok=True)\n    for name in ("models", "results"):\n        shutil.copytree(run / name, bundle / name, dirs_exist_ok=True)\n    shutil.copytree(clip_assets, bundle / "encoders/clip", dirs_exist_ok=True)\n    shutil.copytree(text_assets, bundle / "encoders/distilbert", dirs_exist_ok=True)\n    write_json(config, bundle / "config.json")\n    shutil.copy2(run / "selection.json", bundle / "selection.json")\n    for name in ("clip_models.py", "clip_inference.py", "clip_requirements.txt", "CLIP_EXPERIMENT.md"):\n        shutil.copy2(Path(source_dir) / name, destination / name)\n    return bundle\n', 'clip_inference.py': '"""CPU-only loader for a separate clip_artifacts bundle. Does not touch Streamlit."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\nfrom PIL import Image\nfrom clip_models import ClipEncoder, ClipHead, CachedTextEncoder, FEATURE_SPEC\n\n\nclass ClipPredictor:\n    def __init__(self, bundle="clip_artifacts"):\n        bundle = Path(bundle)\n        self.config = json.loads((bundle / "config.json").read_text(encoding="utf-8"))\n        if self.config["labels"] != {"0": "Fake", "1": "Real"} or self.config["feature_spec"] != FEATURE_SPEC:\n            raise ValueError("Incompatible experimental bundle")\n        self.clip = ClipEncoder(bundle / "encoders/clip", device="cpu", local=True)\n        self.text = CachedTextEncoder(bundle / "encoders/distilbert", device="cpu")\n        self.heads = {}\n        for kind in ("clip_image", "clip_fusion"):\n            saved = torch.load(bundle / "models" / f"{kind}.pt", map_location="cpu", weights_only=True)\n            if saved["run_id"] != self.config["run_id"] or saved["spec"] != self.config["heads"][kind]:\n                raise ValueError("Mixed experiment checkpoints")\n            model = ClipHead(**saved["spec"]).eval()\n            model.load_state_dict(saved["state_dict"])\n            self.heads[kind] = model\n\n    @torch.inference_mode()\n    def predict(self, title, image):\n        if not title.strip():\n            raise ValueError("Enter nonempty text for fusion")\n        text = self.text.encode([title.strip()])\n        visual = self.clip.encode([image])\n        return {kind: model(text, visual).softmax(1)[0].tolist() for kind, model in self.heads.items()}\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--bundle", default="clip_artifacts")\n    parser.add_argument("--text", required=True)\n    parser.add_argument("--image", required=True)\n    args = parser.parse_args()\n    with Image.open(args.image) as image:\n        print(json.dumps(ClipPredictor(args.bundle).predict(args.text, image), indent=2))\n', 'clip_requirements.txt': '# Use a separate environment for the experiment; do not upgrade the working app.\n# Windows: install a matching CPU torch/torchvision pair first (see CLIP_EXPERIMENT.md).\ntorch>=2.6,<3\ntransformers==4.57.1\nsafetensors>=0.4\nPillow>=10,<13\nnumpy>=1.26,<3\n# Training/evaluation only; CPU inference does not need these two libraries.\npandas>=2.2,<3\nscikit-learn>=1.5,<2\nmatplotlib>=3.8,<4\n', 'CLIP_EXPERIMENT.md': '# Frozen CLIP Vision experiment\n\nThis is a separate experiment. `app.py`, `windows_demo/app.py`, the original Colab notebook, baseline modules, processed CSVs, and existing trained checkpoints are not modified. The new notebook is `colab_clip_experiment.ipynb`; it embeds its source files and needs no repository upload.\n\n## Execute in Colab\n\n1. Upload `colab_clip_experiment.ipynb` to Colab and select a GPU runtime. Run the numbered cells in order.\n2. In cell 3, select the existing processed data in Drive, or upload the existing `fakeddit_processed.zip`. No Fakeddit data or images will be downloaded or regenerated.\n3. Point `BASELINE_ROOT` to the original `MyDrive/Fakeddit_Demo` output and `BASELINE_RUN` to the run whose heads you want to compare. The default run ID is the one found in this project\'s trained `windows_demo/artifacts/config.json`: `27751e2dbfb636fc5929`. Set `BASELINE_CACHE_ROOT` to the original `features` directory and `TEXT_ASSETS` to the original `encoders_v1/distilbert` directory. You may point `BASELINE_RUN` at an uploaded baseline artifacts folder instead, but the original feature caches are still required.\n4. Keep `OUTPUT` separate, by default `MyDrive/Fakeddit_CLIP_Experiment`. The notebook refuses output paths overlapping baseline/data directories. It reads the original text and ResNet50 caches without changing them. Missing caches produce an actionable error rather than silently re-extracting or downloading data.\n5. After dataset/cache checks, run frozen CLIP extraction, the validation-only head sweep, validation comparison, then test comparison. Finish with export and the CPU smoke check. Download `clip_windows_<run_id>.zip` from Drive.\n\nRequired original Drive files:\n\n```text\nFakeddit_Demo/\n  runs/27751e2dbfb636fc5929/config.json\n  runs/27751e2dbfb636fc5929/models/{text,image,fusion}.pt\n  features/<feature_id>/{train,validate,test}/{text,image}/*.pt\n  encoders_v1/distilbert/...\n```\n\nThe notebook verifies the original data fingerprint, label convention, checkpoint run IDs, and cache sample IDs/labels/dimensions. Features are reordered by sample ID into CSV order; conflicting duplicates, missing IDs, or extra IDs stop the run. A sample text-encoder check verifies that the supplied DistilBERT assets agree with the cached embeddings before training. All 3,000/500/500 rows stay in their original splits.\n\n## Representation and model selection\n\nUse `openai/clip-vit-base-patch32` with `CLIPVisionModelWithProjection`, taking **512-dimensional `image_embeds`**, followed by L2 normalization. This is distinct from CLIP\'s 768-dimensional pooled hidden representation and ResNet50\'s 2048 features. The head dimensions are read from tensors and serialized with each checkpoint. Image head: 512 → 256 → ReLU → dropout → 2. Fusion: cached DistilBERT 768 → 256 and CLIP 512 → 256, concatenate → 128 → ReLU → dropout → 2.\n\nAll CLIP parameters, including its pretrained projection, stay frozen and in evaluation mode. Only heads train. The pretrained revision is resolved to an immutable commit on first use and recorded with asset checksums. CLIP\'s own saved `CLIPImageProcessor` performs required resize/crop/rescaling/normalization on the **existing processed images**. This does not rewrite images or change the baseline ResNet preprocessing. DistilBERT uses the original tokenizer, max length 128, and attention-mask-aware mean pooling without new text cleaning.\n\nDefault sweep: learning rates 0.001 and 0.0003, dropout 0.3, 25 epochs maximum, batch size 64, seed 42, patience 5, AdamW weight decay 0.0001. The highest **validation macro-F1** selects each epoch/trial; exact ties use lower validation cross-entropy, then fixed grid order. Training never accesses test tensors. Set the grid and minimum promotion gain before running; do not retune after inspecting test results.\n\nThe same five models are evaluated on the same validation/test pairs: DistilBERT reference, ResNet50, DistilBERT + ResNet50, CLIP image-only, and DistilBERT + CLIP. Both splits receive accuracy, precision, recall, F1, macro-F1, ROC-AUC and confusion matrices, plus per-sample predictions and comparison charts. Precision/recall/F1/ROC-AUC treat **Fake (0)** as positive; Real is 1. Macro-F1 averages both classes equally. Matrix rows are actual, columns predicted, ordered Fake/Real. Undefined AUC is recorded as null for a one-class fixture, never fabricated.\n\n`results/promotion.json` uses **validation only**: CLIP image-only must strictly exceed ResNet50 macro-F1, and CLIP fusion must strictly exceed baseline fusion macro-F1. `MINIMUM_VALIDATION_GAIN` can require a larger predeclared gain. These are observed improvements on 500 validation samples, not proof of statistical significance. Test results are reported regardless of whether CLIP improves validation; they never change eligibility. There are no claimed CLIP scores until the notebook runs.\n\n## Recovery and output layout\n\n```text\nFakeddit_CLIP_Experiment/\n  encoders/<source-key>/                 # Frozen CLIP, processor, revision/checksums\n  features/<fingerprint>/<split>/*.pt    # New CLIP embeddings only\n  runs/<experiment-id>/\n    config.json, selection.json\n    trials/<head>/<trial-id>/           # Last + best state, optimizer, RNG, history\n    models/{clip_image,clip_fusion}.pt\n    results/                            # Validation/test comparisons and promotion gate\n  clip_windows_<experiment-id>.zip      # Separate CPU bundle\n```\n\nRerun all cells with the same paths/settings after interruption. Completed feature shards are reused even if `FEATURE_BATCH_SIZE` is reduced; at most the incomplete 128-example shard is repeated. Training resumes from the last atomically saved epoch, including best weights, optimizer, RNG and early-stopping counters. Finished trials skip training. Input data, encoder assets, cached baseline tensors, source code, and experiment settings contribute to fingerprints so incompatible runs are separated. Partial `.tmp` files are ignored. Do not run two sessions against the same experiment output simultaneously.\n\n## Windows CPU export (no app changes)\n\nExtract the experimental ZIP into a **new folder**, such as `clip_demo`. It contains only `clip_artifacts/`, experimental Python modules, requirements and this guide. It has no `app.py` and cannot replace the working Streamlit application. Both frozen encoders are included, and inference uses local files only with `map_location="cpu"`.\n\nCreate a separate environment; leave the existing working app environment alone:\n\n```powershell\npy -3.12 -m venv .venv-clip\n.\\.venv-clip\\Scripts\\python.exe -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu\n.\\.venv-clip\\Scripts\\python.exe -m pip install -r clip_requirements.txt\n.\\.venv-clip\\Scripts\\python.exe clip_inference.py --bundle clip_artifacts --text "Example headline" --image "C:\\path\\to\\image.jpg"\n```\n\nColab verifies an exported CPU prediction against cached GPU features for both heads before publishing the ZIP. `clip_artifacts/results/cpu_smoke.json` records this check. The export also records dependency versions and a file checksum manifest. This checks CPU serialization and prediction parity on Colab; run the above command on Windows to verify its actual runtime too.\n\n## Conditional Streamlit integration instructions\n\nOnly consider integration after `promotion.json` marks the particular model `eligible: true`, evaluation completes, and the exported CPU smoke check passes. If a candidate is ineligible, keep the existing app and checkpoint selection unchanged. The notebook also generates `INTEGRATION_DECISION.md` listing the eligible candidates.\n\nFor eligible candidates, first copy the working app to a separate evaluation folder. Keep its baseline `artifacts/` and add the experimental `clip_artifacts/` alongside it. Copy `clip_models.py` and `clip_inference.py` from the experimental export. Do not replace the baseline loader or its model classes: their feature dimensions and checkpoint keys differ.\n\nIn that **copy**, add a separately cached loader and an explicit experimental choice. A future integration can use:\n\n```python\nfrom pathlib import Path\nfrom clip_inference import ClipPredictor\n\n@st.cache_resource(max_entries=1)\ndef load_clip(bundle_path):\n    return ClipPredictor(bundle_path)\n\n# Inside the existing Analyze handler, after validating the headline and image:\nclip_bundle = Path(__file__).resolve().parent / "clip_artifacts"\nclip_scores = load_clip(str(clip_bundle)).predict(title, image)\n# clip_scores["clip_image"] and clip_scores["clip_fusion"] are [P(Fake), P(Real)].\n# Display only the model(s) marked eligible in promotion.json, with an experimental label.\n```\n\nRetain all existing baseline predictions and comparison views. Show the separate CLIP validation/test report rather than overwriting `artifacts/results/metrics.json`. Keep the new model optional until Windows single-example inference and memory/latency are acceptable. Clear the model resource cache or restart the copied app after changing bundles. No integration code has been applied to either existing app.\n\n## Development checks\n\nRun `python build_clip_notebook.py` after editing experimental sources, then `python -m unittest discover -s tests -p "test_clip*.py" -v` in an environment with the experiment requirements and torchvision. Tests use small synthetic tensors and tiny randomly initialized CLIP configs to verify mechanics without downloading pretrained weights or reporting fixture metrics as experiment results. Full pretrained extraction/training remains a Colab GPU task.\n\nReferences: [Transformers CLIP documentation](https://huggingface.co/docs/transformers/model_doc/clip), [OpenAI CLIP model](https://huggingface.co/openai/clip-vit-base-patch32). The CLIP processor and projected feature API follow these sources.\n'}
for name, source in SOURCE_FILES.items():
    (WORK / name).write_text(source, encoding='utf-8')


## 3. Set paths and validation-only sweep
Use the original Drive outputs from colab_training.ipynb. The known baseline run ID below comes from this project's trained Windows bundle; edit if using a different run.
The original features folder is required for both cached DistilBERT reuse and fair ResNet50 comparison. A Windows export alone does not contain these feature caches.
Set the hyperparameter grid and promotion gain **before** looking at test results. Do not change the working app.


In [ ]:
DATA_SOURCE = "drive"  # "drive" or "zip"
DATA_DIR = Path('/content/drive/MyDrive/Multimodal_Fake_News_Detection/data')
ZIP_ON_DRIVE = ""  # Optional existing processed ZIP; otherwise upload when DATA_SOURCE == "zip".
BASELINE_ROOT = Path('/content/drive/MyDrive/Fakeddit_Demo')
BASELINE_RUN = BASELINE_ROOT / 'runs/27751e2dbfb636fc5929'
BASELINE_CACHE_ROOT = BASELINE_ROOT / 'features'
TEXT_ASSETS = BASELINE_ROOT / 'encoders_v1/distilbert'
OUTPUT = Path('/content/drive/MyDrive/Fakeddit_CLIP_Experiment')
CLIP_REVISION = "main"  # Resolved once to an immutable commit and saved for recovery.
FEATURE_BATCH_SIZE = 32  # Can reduce without losing completed fixed-size feature shards.
TRIAL_CONFIGS = [dict(seed=42, epochs=25, batch_size=64, lr=lr, weight_decay=0.0001,
                      patience=5, dropout=0.3) for lr in (0.001, 0.0003)]
MINIMUM_VALIDATION_GAIN = 0.0  # Strict macro-F1 gain above the corresponding baseline.
from dataset import extract_zip, validate
from clip_experiment import guard_output
guard_output(OUTPUT, BASELINE_ROOT, BASELINE_RUN, BASELINE_CACHE_ROOT, TEXT_ASSETS, DATA_DIR)
OUTPUT.mkdir(parents=True, exist_ok=True)
if DATA_SOURCE == 'zip':
    if ZIP_ON_DRIVE:
        archive = Path(ZIP_ON_DRIVE)
    else:
        uploaded = files.upload()
        assert len(uploaded) == 1, 'Upload exactly one processed dataset ZIP.'
        archive = WORK / next(iter(uploaded))
        del uploaded
    DATA_DIR = extract_zip(archive, Path('/content/fakeddit_clip_data'))
elif DATA_SOURCE != 'drive':
    raise ValueError('Choose drive or zip')
guard_output(OUTPUT, DATA_DIR)
assert (BASELINE_RUN / 'config.json').is_file(), f'Choose the existing baseline run: {BASELINE_RUN}'
assert (TEXT_ASSETS / 'config.json').is_file(), f'Missing original DistilBERT assets: {TEXT_ASSETS}'


## 4. Validate unchanged data and load existing caches by sample ID
Checks all 3,000/500/500 records/images, labels (0=Fake, 1=Real), duplicates and overlaps. No rows are dropped. Windows/Linux image paths are resolved without editing CSVs.
Baseline fingerprints must match the dataset. Cached text and ResNet50 embeddings are joined by ID, not by file order. Missing/inconsistent caches stop the experiment.


In [ ]:
from clip_experiment import (read_baseline_config, load_baseline_features, cache_hashes,
    prepare_clip, extract_clip, paired_features, digest, sha_file, select_trials,
    compare_models, promotion_gate, plot_comparison, export_bundle)
from pipeline import fingerprint, write_json
from clip_models import FEATURE_SPEC, CachedTextEncoder
frames, validation_report = validate(DATA_DIR, OUTPUT / 'dataset_validation.json')
baseline_config, baseline_provenance = read_baseline_config(BASELINE_RUN, frames)
baseline = load_baseline_features(BASELINE_CACHE_ROOT, baseline_config['feature_id'], frames)
baseline_provenance['cache_sha256'] = cache_hashes(baseline)
baseline_provenance['text_asset_sha256'] = {p.relative_to(TEXT_ASSETS).as_posix(): sha_file(p)
    for p in TEXT_ASSETS.rglob('*') if p.is_file()}
print({s: len(d['ids']) for s, d in baseline.items()})
# Verify that exported text encoder assets actually reproduce the cached text representation.
text_encoder = CachedTextEncoder(TEXT_ASSETS, device='cuda')
with torch.inference_mode():
    actual = text_encoder.encode(frames['validate'].clean_title.iloc[:2].tolist()).cpu()
torch.testing.assert_close(actual, baseline['validate']['text'][:2], atol=2e-3, rtol=2e-3)
del text_encoder
gc.collect()
torch.cuda.empty_cache()
print('Original data, labels, caches and text encoder verified.')


## 5. Load frozen CLIP Vision and extract resumable image embeddings
Uses CLIPVisionModelWithProjection.image_embeds, 512-D, L2-normalized. The saved CLIPImageProcessor applies model-required transforms to existing RGB images. Baseline preprocessing is unchanged.
All encoder parameters, including the pretrained projection, are frozen. Completed 128-example shards survive runtime disconnections; only an incomplete shard is repeated.


In [ ]:
clip_encoder, CLIP_ASSETS, clip_manifest = prepare_clip(OUTPUT, CLIP_REVISION)
assert not clip_encoder.model.training
assert all(not p.requires_grad for p in clip_encoder.model.parameters())
assert clip_encoder.dimension == 512
feature_contract = {'feature_spec': FEATURE_SPEC, 'clip_assets': clip_manifest,
                    'encoder_code_sha256': hashlib.sha256(SOURCE_FILES['clip_models.py'].encode()).hexdigest()}
CLIP_FEATURE_ID = fingerprint(frames, feature_contract)
visual = extract_clip(frames, clip_encoder, OUTPUT / 'features', CLIP_FEATURE_ID, FEATURE_BATCH_SIZE)
clip_data = paired_features(baseline, visual)
for split, frame in frames.items():
    assert clip_data[split]['image'].shape == (len(frame), 512)
    assert clip_data[split]['text'].shape == (len(frame), 768)
    assert clip_data[split]['ids'] == frame.id.tolist()
del clip_encoder
gc.collect()
torch.cuda.empty_cache()
run_contract = {'version': 1, 'clip_feature_id': CLIP_FEATURE_ID, 'baseline': baseline_provenance,
                'trials': TRIAL_CONFIGS, 'minimum_validation_gain': MINIMUM_VALIDATION_GAIN,
                'source_sha256': {name: hashlib.sha256(source.encode()).hexdigest()
                                  for name, source in SOURCE_FILES.items() if name.endswith('.py')}}
RUN_ID = digest(run_contract)
RUN = OUTPUT / 'runs' / RUN_ID
RUN.mkdir(parents=True, exist_ok=True)
write_json(validation_report, RUN / 'results/dataset_validation.json')
write_json(run_contract, RUN / 'provenance.json')
print('New isolated experiment run:', RUN)


## 6. Train image-only and fusion heads; select on validation macro-F1
Only train tensors update weights. Validation macro-F1 selects checkpoints/trials (loss breaks ties).
This cell passes **no test data** to training. Resume restores last/best weights, AdamW state, RNG and patience. Completed trials are reused.


In [ ]:
training_data = {s: clip_data[s] for s in ('train', 'validate')}
selected = select_trials(training_data, RUN, TRIAL_CONFIGS, RUN_ID)
config = {'run_id': RUN_ID, 'feature_spec': FEATURE_SPEC, 'labels': {'0': 'Fake', '1': 'Real'},
          'heads': {kind: info['spec'] for kind, info in selected.items()},
          'clip_revision': clip_manifest['resolved_revision'], 'clip_feature_id': CLIP_FEATURE_ID,
          'baseline_run_id': baseline_config['run_id'], 'text_encoder': baseline_config['encoders'],
          'selection_metric': 'validation macro-F1', 'provenance': run_contract}
write_json(config, RUN / 'config.json')
print(json.dumps(selected, indent=2))


## 7. Compare validation scores and lock the integration decision
Re-evaluates the original three checkpoints without retraining them, adding macro-F1. CLIP image-only is compared to ResNet50; CLIP fusion to DistilBERT + ResNet50.
Eligibility requires strict validation macro-F1 improvement above the configured minimum. No claim of statistical significance. The test set cannot influence this decision.


In [ ]:
validation = compare_models(baseline, clip_data, BASELINE_RUN, RUN, RUN_ID, splits=('validate',))['validate']
gate = promotion_gate(validation, MINIMUM_VALIDATION_GAIN)
write_json(gate, RUN / 'results/promotion.json')
decision_lines = ['# Validation-based integration decision', '', gate['criterion'], '']
for kind, decision in gate['candidates'].items():
    status = 'Eligible for an optional integration trial' if decision['eligible'] else 'Not eligible; keep the existing model'
    decision_lines.append(f"- {kind}: {status}. Validation macro-F1 change: {decision['delta_macro_f1']:+.6f}.")
decision_lines += ['', 'See CLIP_EXPERIMENT.md for conditional integration steps. No Streamlit app has been modified.',
                   'Test results must not be used to change this validation-based selection.']
(RUN / 'results/INTEGRATION_DECISION.md').write_text('\n'.join(decision_lines), encoding='utf-8')
plot_comparison(RUN, 'validate')
import pandas as pd
from IPython.display import display, Image as DisplayImage
display(pd.DataFrame(validation['models']).T)
display(DisplayImage(filename=str(RUN / 'results/validate_comparison.png')))
print(json.dumps(gate, indent=2))


## 8. Final comparison on the same 500 test examples
All heads are already selected. Reports accuracy, Fake-positive precision/recall/F1, macro-F1, ROC-AUC and confusion matrices (Fake, Real).
Always reports test results, even when validation did not improve. Do not tune against these results.


In [ ]:
assert (RUN / 'selection.json').exists() and (RUN / 'results/promotion.json').exists()
test = compare_models(baseline, clip_data, BASELINE_RUN, RUN, RUN_ID, splits=('test',))['test']
plot_comparison(RUN, 'test')
write_json({'validation': validation, 'test': test, 'promotion': gate}, RUN / 'results/metrics.json')
display(pd.DataFrame(test['models']).T)
display(DisplayImage(filename=str(RUN / 'results/test_comparison.png')))
display(DisplayImage(filename=str(RUN / 'results/test_confusion_matrices.png')))


## 9. Build a separate CPU export and verify prediction parity
Exports frozen CLIP + its processor, original DistilBERT, selected experimental heads/configs/results, and an offline CPU loader. Contains no app.py or production artifacts folder.
CPU predictions must agree with cached GPU-feature predictions before the ZIP is published. This is a single-pair compatibility check, not CPU training.


In [ ]:
from clip_inference import ClipPredictor
from PIL import Image
import importlib.metadata
import zipfile
EXPORT_DIR = Path('/content/clip_exports') / RUN_ID
BUNDLE = export_bundle(RUN, CLIP_ASSETS, TEXT_ASSETS, EXPORT_DIR, config, WORK)
shutil.copy2(RUN / 'provenance.json', BUNDLE / 'provenance.json')
versions = {p: importlib.metadata.version(p) for p in ('torch', 'torchvision', 'transformers', 'numpy', 'pandas', 'Pillow', 'scikit-learn')}
write_json(versions, BUNDLE / 'environment.json')
torch.set_num_threads(2)
predictor = ClipPredictor(BUNDLE)
row = frames['test'].iloc[0]
with Image.open(row.resolved_path) as image:
    predictions = predictor.predict(row.clean_title, image)
for kind, head in predictor.heads.items():
    with torch.inference_mode():
        expected = head(clip_data['test']['text'][:1], clip_data['test']['image'][:1]).softmax(1)[0]
    torch.testing.assert_close(torch.tensor(predictions[kind]), expected, atol=2e-3, rtol=2e-3)
smoke = {'passed': True, 'sample_id': row.id, 'probabilities': predictions,
         'check': 'offline CPU vs cached GPU features; atol=rtol=0.002'}
write_json(smoke, BUNDLE / 'results/cpu_smoke.json')
write_json(smoke, RUN / 'results/cpu_smoke.json')
# Check that no baseline checkpoint changed during the experiment.
for kind, expected_hash in baseline_provenance['checkpoint_sha256'].items():
    assert sha_file(BASELINE_RUN / 'models' / f'{kind}.pt') == expected_hash
manifest = {p.relative_to(EXPORT_DIR).as_posix(): sha_file(p) for p in EXPORT_DIR.rglob('*')
            if p.is_file() and p.name != 'SHA256SUMS.json'}
write_json(manifest, EXPORT_DIR / 'SHA256SUMS.json')
export_path = OUTPUT / f'clip_windows_{RUN_ID}.zip'
temporary = export_path.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in EXPORT_DIR.rglob('*'):
        if path.is_file():
            archive.write(path, path.relative_to(EXPORT_DIR).as_posix())
os.replace(temporary, export_path)
print('CPU check passed. Download the separate bundle from Drive:', export_path)
print('The working Streamlit app and baseline checkpoints remain unchanged.')


## 10. Download and review
Prefer downloading the ZIP from Google Drive. Extract into a new Windows folder; follow CLIP_EXPERIMENT.md to run CPU inference.
Only models marked eligible in results/promotion.json should be considered for optional integration into a copy of the app. No integration happens automatically.
After a disconnect, rerun cells sequentially with identical settings; persistent shards and trial recovery are reused.


In [ ]:
# Optional browser download (the ZIP is already saved in Drive):
# files.download(str(export_path))
